In [ ]:
# Instalaivimas reikalingų paketų
#!git clone https://github.com/deepseek-ai/DeepSeek-VL
%cd DeepSeek-VL
!pip install -e .
print("Instaliavimas baigtas")

In [ ]:
# Perspėjimų ignoravimas
import warnings
warnings.filterwarnings("ignore")  # Ignore all warnings

from transformers import AutoModelForCausalLM
from deepseek_vl.models import VLChatProcessor, MultiModalityCausalLM
from deepseek_vl.utils.io import load_pil_images
from PIL import Image
import requests
import torch
import os
import re
import pandas as pd
print("Importavimas baigtas")

In [ ]:
# Modelių naudojimas
model_path = "deepseek-ai/deepseek-vl-7b-chat"
vl_chat_processor: VLChatProcessor = VLChatProcessor.from_pretrained(model_path)
tokenizer = vl_chat_processor.tokenizer

vl_gpt: MultiModalityCausalLM = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True)
vl_gpt = vl_gpt.to(torch.bfloat16).cuda().eval()

print("Modelio krovimas baigtas")

In [ ]:
# Lentelės užkrovimas, generavimas ir saugojimas
df = pd.read_csv('../flicker8k_edited.csv', sep=';')
import numpy as np

# Išsivalome lentelę
columns_to_clear = [
    'BLEU', 'ROGUE', 'METEOR',
    'BERTscore Precision', 'BERTscore Recall', 'BERTscore F1', 'SentenceBERT'
]
for col in columns_to_clear:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: '')

image_folder = "../Flickr8k"

#for index, row in df.head(20).iterrows():
for index, row in df.iterrows():
    filename = row["Image Name"]
    image_path = os.path.join(image_folder, filename)
    image = Image.open(image_path).convert("RGB")
    try:
        image = Image.open(image_path)
        conversation = [
                {
                    "role": "User",
                    "content": "<image_placeholder>Keep it short. No more than 10 words. The text is the caption for the image. Do not write any additional text.",
                    "images": [image_path]
                },
                {
                    "role": "Assistant",
                    "content": ""
                }
        ]

        pil_images = load_pil_images(conversation)
        prepare_inputs = vl_chat_processor(
            conversations=conversation,
            images=pil_images,
            force_batchify=True
            ).to(vl_gpt.device)

        inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)

        outputs = vl_gpt.language_model.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=prepare_inputs.attention_mask,
            pad_token_id=tokenizer.eos_token_id,
            bos_token_id=tokenizer.bos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            max_new_tokens=50,
            do_sample=False,
            use_cache=True
        )

        generated_caption = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)

        # Įrašymas į failą
        df.at[index, "Generated Caption"] = generated_caption

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        df.at[index, "Generated Caption"] = f"Error: {e}"

# Saugojimas
df.to_csv("../Results/flicker8k_en_19.csv", sep=';', index=False)
print("Generavimas baigtas.")